In [17]:
import polars as pl
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import gc

In [18]:
# ----------------------------------------------------------------------
# 1. Загрузка данных
# ----------------------------------------------------------------------
print("Загрузка основных данных...")
train = pl.read_parquet('data/train_main_features.parquet')
test = pl.read_parquet('data/test_main_features.parquet')
target = pl.read_parquet('data/train_target.parquet')

Загрузка основных данных...


In [20]:
# ----------------------------------------------------------------------
# 2. Отбор важных дополнительных признаков
# ----------------------------------------------------------------------
USE_EXTRA = True
EXTRA_FEATURES_N = 200

if USE_EXTRA:
    print("Загрузка дополнительных данных (только для отбора признаков)...")
    sample_size = min(100_000, train.height)
    train_extra_sample = pl.read_parquet('data/train_extra_features.parquet').sample(n=sample_size, seed=42)
    
    # Присоединяем таргеты
    target_sample = target.join(train_extra_sample.select('customer_id'), on='customer_id')
    target_sample_pd = target_sample.drop('customer_id').to_pandas()
    
    # Определяем колонки в extra
    extra_cat_cols = [c for c in train_extra_sample.columns if c.startswith('cat_feature')]
    extra_num_cols = [c for c in train_extra_sample.columns if c.startswith('num_feature')]
    
    # ---- Отбор числовых признаков ----
    if len(extra_num_cols) > 0:
        extra_num_df = train_extra_sample.select(extra_num_cols).to_pandas()
        extra_num_df = extra_num_df.fillna(extra_num_df.median())
        corr_matrix = extra_num_df.corrwith(target_sample_pd, axis=0)
        if isinstance(corr_matrix, pd.Series):
            top_num_extra = corr_matrix.abs().sort_values(ascending=False).head(EXTRA_FEATURES_N).index.tolist()
        else:
            corr_mean = corr_matrix.abs().mean(axis=1).sort_values(ascending=False)
            top_num_extra = corr_mean.head(EXTRA_FEATURES_N).index.tolist()
        print(f"Отобрано {len(top_num_extra)} числовых extra признаков")
    else:
        top_num_extra = []
    
    # ---- Отбор категориальных признаков ----
    if len(extra_cat_cols) > 0:
        unique_counts = {}
        for col in extra_cat_cols:
            uniq = train_extra_sample[col].drop_nulls().unique().len()
            unique_counts[col] = uniq
        top_cat_extra = [col for col, cnt in unique_counts.items() if 1 < cnt < 1000]
        if len(top_cat_extra) > 500:
            top_cat_extra = top_cat_extra[:500]
        print(f"Отобрано {len(top_cat_extra)} категориальных extra признаков")
    else:
        top_cat_extra = []
    
    selected_extra_features = top_num_extra + top_cat_extra
    print(f"Всего отобрано extra признаков: {len(selected_extra_features)}")
    
    if selected_extra_features:
        train_extra = pl.read_parquet('data/train_extra_features.parquet', columns=['customer_id'] + selected_extra_features)
        test_extra = pl.read_parquet('data/test_extra_features.parquet', columns=['customer_id'] + selected_extra_features)
        extra_cat_in_selected = [c for c in selected_extra_features if c.startswith('cat_feature')]
        train_extra = train_extra.with_columns(pl.col(extra_cat_in_selected).cast(pl.Int32))
        test_extra = test_extra.with_columns(pl.col(extra_cat_in_selected).cast(pl.Int32))
        train = train.join(train_extra, on='customer_id', how='left')
        test = test.join(test_extra, on='customer_id', how='left')
        del train_extra, test_extra
        gc.collect()
    else:
        print("Не найдено подходящих extra признаков, пропускаем")
else:
    selected_extra_features = []


Загрузка дополнительных данных (только для отбора признаков)...
Отобрано 200 числовых extra признаков
Всего отобрано extra признаков: 200


In [22]:
# ----------------------------------------------------------------------
# 3. Предобработка
# ----------------------------------------------------------------------
# Определяем категориальные признаки
cat_features = [col for col in train.columns if col.startswith("cat_feature")]
train = train.with_columns(pl.col(cat_features).cast(pl.Int32))
test = test.with_columns(pl.col(cat_features).cast(pl.Int32))

# Добавляем индикаторы пропусков для числовых признаков
num_features = [col for col in train.columns if col.startswith("num_feature")]
for col in num_features:
    train = train.with_columns(pl.col(col).is_null().cast(pl.Int32).alias(f"{col}_isna"))
    test = test.with_columns(pl.col(col).is_null().cast(pl.Int32).alias(f"{col}_isna"))
    cat_features.append(f"{col}_isna")

# Отделяем customer_id
train_ids = train['customer_id']
train_features = train.drop('customer_id')
test_ids = test['customer_id']
test_features = test.drop('customer_id')

# Целевые переменные
target_cols = [col for col in target.columns if col.startswith("target")]
target_data = target.select(target_cols).to_pandas()

In [23]:
# ----------------------------------------------------------------------
# 4. Разделение на обучающую и валидационную выборки
# ----------------------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    train_features.to_pandas(),
    target_data,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [24]:
# ----------------------------------------------------------------------
# 5. Обучение отдельных бинарных моделей для каждого класса
# ----------------------------------------------------------------------
print("\nОбучение бинарных моделей (one-vs-rest) для 41 класса...")
models = {}
val_auc = {}

for i, target_col in enumerate(target_cols):
    print(f"\nОбучаем модель {i+1}/{len(target_cols)}: {target_col}")
    
    train_pool = Pool(
        X_train,
        label=y_train[target_col],
        cat_features=cat_features
    )
    val_pool = Pool(
        X_val,
        label=y_val[target_col],
        cat_features=cat_features
    )
    
    model = CatBoostClassifier(
        iterations=500,               # увеличено
        depth=6,
        learning_rate=0.03,
        loss_function='Logloss',
        eval_metric='AUC',
        l2_leaf_reg=3,
        random_seed=42,
        verbose=False,
        early_stopping_rounds=50,
        task_type='CPU',
        thread_count=4
    )
    
    model.fit(
        train_pool,
        eval_set=val_pool,
        plot=False
    )
    
    pred_val = model.predict_proba(val_pool)[:, 1]
    auc = roc_auc_score(y_val[target_col], pred_val)
    val_auc[target_col] = auc
    models[target_col] = model
    print(f"  ROC-AUC на валидации: {auc:.4f}")


Обучение бинарных моделей (one-vs-rest) для 41 класса...

Обучаем модель 1/41: target_1_1


KeyboardInterrupt: 

In [ ]:
# ----------------------------------------------------------------------
# 6. Предсказание на тестовых данных
# ----------------------------------------------------------------------
print("\nФормирование предсказаний для тестовой выборки...")
test_pool = Pool(test_features.to_pandas(), cat_features=cat_features)

predictions = []
for target_col in target_cols:
    model = models[target_col]
    proba = model.predict_proba(test_pool)[:, 1]
    predictions.append(proba)

predictions = np.column_stack(predictions)
predict_cols = [f"predict_{col.replace('target_', '')}" for col in target_cols]
pred_df = pl.DataFrame(predictions, schema=predict_cols)

In [ ]:
# ----------------------------------------------------------------------
# 7. Сохранение сабмита
# ----------------------------------------------------------------------
submit = pl.DataFrame({'customer_id': test_ids}).hstack(pred_df)
submit.write_parquet("data/submit.parquet")
print("\nСабмит сохранён в 'data/submit.parquet'")

mean_auc = np.mean(list(val_auc.values()))
print(f"\nСредний ROC-AUC (macro) на валидации: {mean_auc:.4f}")